# 05 - Algoritma Karşılaştırması: Nicel (Quantitative) Analiz

**Amaç:** `Model_Base`, `Model_SFT`, `Model_RLHF` (PPO) ve `Model_DPO`
modellerini nicel metriklerle karşılaştırmak:

1. Eğitim kayıp (loss) eğrileri (SFT, Ödül Modeli, PPO, DPO)
2. Ödül Modeli doğruluğu
3. Ortalama yanıt uzunlukları (**uzunluk istismarı** / length-exploitation tespiti;
   Park vd., 2024; Lambert, 2025)
4. Simge bazlı $D_{KL}(\pi_{\text{politika}} \| \pi_{\text{Model\_SFT}})$ sapma miktarları

> Bu defter, `02_SFT_Training.ipynb` → `03_RLHF_Training.ipynb` →
> `04_DPO_Training.ipynb` defterlerinin ürettiği çıktılara (adaptörler ve log
> dosyaları) ihtiyaç duyar; bunların Google Drive üzerinde `outputs/` altında
> mevcut olduğundan emin olun.

## 0. Ortam Kurulumu

In [ ]:
# --- Google Colab ortam kurulumu -------------------------------------------
# Bu hucre, projenin tamamini (Datasets/, src/, notebooks/, outputs/) iceren
# "Uygulama" klasorunu Google Drive'a yukledikten sonra calistirilmalidir.
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # TODO: Kendi Google Drive yolunuza gore guncelleyin
    PROJECT_ROOT = "/content/drive/MyDrive/Uygulama"
else:
    PROJECT_ROOT = os.path.abspath("..")

SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IN_COLAB:", IN_COLAB)

In [ ]:
# NOT: Surumler, TRL'nin klasik `PPOTrainer` + `AutoModelForCausalLMWithValueHead`
# coklu-adaptor (multi-adapter) RL ozelligiyle dogrulanmis, bilinen-uyumlu bir
# kombinasyon olacak sekilde SABITLENMISTIR (API kararliligi icin).
%pip install -q -U "transformers==4.44.2" "trl==0.9.6" "peft==0.11.1" "bitsandbytes==0.43.1" "accelerate==0.32.1" "datasets==2.20.0" "numpy==1.26.4" sentencepiece

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import alignment_pipeline as ap

ap.set_global_seed(42)
print("Cihaz:", ap.get_device())

In [ ]:
config = ap.ExperimentConfig(
    dataset_root=os.path.join(PROJECT_ROOT, "Datasets"),
    output_root=os.path.join(PROJECT_ROOT, "outputs"),
)
ap.ensure_dir(config.output_root)
print("Taban model:", config.base_model_id)
print("Veri kok dizini:", config.dataset_root)
print("Cikti kok dizini:", config.output_root)

In [ ]:
SFT_MERGED_DIR = config.path("sft", "merged_model")
SFT_LOGS_DIR = config.path("sft", "logs")
RM_LOGS_DIR = config.path("reward_model", "logs")
PPO_ADAPTER_DIR = config.path("ppo", "ppo_adapter")
PPO_HISTORY_CSV = config.path("ppo", "ppo_training_history.csv")
DPO_ADAPTER_DIR = config.path("dpo", "adapter")
DPO_LOGS_DIR = config.path("dpo", "logs")

QUANT_BITS = 4
TEST_PROMPTS = ap.DEFAULT_QUALITATIVE_TEST_PROMPTS

## 1. Eğitim Kayıp (Loss) Eğrileri

In [ ]:
sft_log_df = pd.DataFrame(ap.load_json(os.path.join(SFT_LOGS_DIR, "sft_log_history.json")))
dpo_log_df = pd.DataFrame(ap.load_json(os.path.join(DPO_LOGS_DIR, "dpo_log_history.json")))

ap.plot_multi_run_losses(
    {"SFT": sft_log_df, "DPO": dpo_log_df},
    loss_col="loss",
    title="SFT vs. DPO - Eğitim Kayıp Karşılaştırması",
)

In [ ]:
rm_log_df = pd.DataFrame(ap.load_json(os.path.join(RM_LOGS_DIR, "reward_log_history.json")))
ap.plot_loss_curve(rm_log_df, loss_col="loss", title="Ödül Modeli (Reward Model) Eğitim Kaybı")

In [ ]:
ppo_history_df = pd.read_csv(PPO_HISTORY_CSV)
ap.plot_ppo_history(ppo_history_df)

## 2. Ödül Modeli Doğruluğu

In [ ]:
rm_eval_metrics = ap.load_json(os.path.join(RM_LOGS_DIR, "reward_eval_metrics.json"))
pd.Series(rm_eval_metrics).to_frame("değer")

## 3. Ortalama Yanıt Uzunlukları (Uzunluk İstismarı Tespiti)

RLHF (PPO) politikalarının, ödül modelini "kandırmak" için gereğinden uzun
yanıtlar üretmeye yönelebileceği bilinen bir olgudur (reward hacking /
length exploitation). Bu bölümde dört modelin aynı test istemlerine verdiği
yanıtların ortalama token uzunlukları karşılaştırılır.

In [ ]:
model_paths = {
    "Model_Base": {"base_model_id": config.base_model_id, "adapter_path": None},
    "Model_SFT":  {"base_model_id": SFT_MERGED_DIR,       "adapter_path": None},
    "Model_RLHF": {"base_model_id": SFT_MERGED_DIR,       "adapter_path": PPO_ADAPTER_DIR},
    "Model_DPO":  {"base_model_id": SFT_MERGED_DIR,       "adapter_path": DPO_ADAPTER_DIR},
}

loaded_models = {}
for name, paths in model_paths.items():
    model, tok = ap.load_model_for_inference(paths["base_model_id"], paths["adapter_path"], quant_bits=QUANT_BITS)
    loaded_models[name] = (model, tok)
    print(f"{name} yüklendi.")

In [ ]:
responses_by_model = {}
length_stats_by_model = {}
for name, (model, tok) in loaded_models.items():
    responses = ap.generate_batch(model, tok, TEST_PROMPTS, max_new_tokens=config.max_new_tokens)
    responses_by_model[name] = responses
    length_stats_by_model[name] = ap.response_length_stats(responses, tok)
    print(f"{name}: ortalama yanıt uzunluğu = {length_stats_by_model[name]['mean']:.1f} token")

ap.plot_bar_comparison(
    {name: stats["mean"] for name, stats in length_stats_by_model.items()},
    title="Ortalama Yanıt Uzunluğu (token)",
    ylabel="Ortalama token sayısı",
)

## 4. Simge Bazlı $D_{KL}$ Sapma Miktarları

Referans politika olarak `Model_SFT` alınır; `Model_RLHF` ve `Model_DPO`'nun
kendi ürettikleri yanıtlar üzerinden referansa göre ortalama token-bazlı KL
sapması hesaplanır (Rafailov vd., 2023; Park vd., 2024).

In [ ]:
reference_model, reference_tok = loaded_models["Model_SFT"]

kl_results = {}
for name in ("Model_RLHF", "Model_DPO"):
    policy_model, policy_tok = loaded_models[name]
    kl_stats = ap.compute_token_kl_divergence(
        policy_model, reference_model, policy_tok, TEST_PROMPTS, max_new_tokens=64
    )
    kl_results[name] = kl_stats["mean_kl_per_token"]
    print(f"{name}: ortalama token-bazlı D_KL(pi || pi_SFT) = {kl_stats['mean_kl_per_token']:.4f}")

ap.plot_bar_comparison(kl_results, title="Simge Bazlı D_KL(politika || Model_SFT)", ylabel="Ortalama D_KL / token")

## 5. Özet Tablo

In [ ]:
summary_rows = []
for name in model_paths:
    summary_rows.append({
        "model": name,
        "ortalama_yanit_uzunlugu": length_stats_by_model[name]["mean"],
        "std_yanit_uzunlugu": length_stats_by_model[name]["std"],
        "ortalama_token_kl_vs_sft": kl_results.get(name, 0.0 if name == "Model_SFT" else None),
    })
summary_df = pd.DataFrame(summary_rows)
display(summary_df)

QUAL_OUT_DIR = config.path("quantitative_summary")
ap.ensure_dir(QUAL_OUT_DIR)
summary_df.to_csv(os.path.join(QUAL_OUT_DIR, "nicel_ozet_tablo.csv"), index=False)

## 6. Gözlemler

- SFT ve DPO kayıp eğrileri yakınsama (convergence) davranışı açısından
  karşılaştırılmalı; PPO'nun ödül/KL eğrileri birlikte yorumlanmalıdır
  (ör. ödül artarken KL'nin aşırı büyümesi, ödül "hackleme" işaretidir;
  Lambert, 2025).
- `Model_RLHF` için ortalama yanıt uzunluğunun `Model_SFT`'ye göre belirgin
  şekilde artması, olası bir **uzunluk istismarı** sinyali olarak
  yorumlanmalıdır.
- Sıradaki adım: `06_Algorithm_Comparison_Qualitative.ipynb`
  (LLM-as-a-Judge ile nitel karşılaştırma).